In [37]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")

print("Database Connected")

Database Connected


In [38]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")

cursor = conn.cursor()

with open("sql/schema.sql", "r") as file:
    sql_script = file.read()

# cursor.executescript(sql_script)
cursor.execute("DROP TABLE IF EXISTS dim_fund")
conn.commit()

conn.commit()

print("Tables Created Successfully")

Tables Created Successfully


In [39]:
cursor.execute("""
SELECT name
FROM sqlite_master
WHERE type='table';
""")

print(cursor.fetchall())

[('dim_date',), ('fact_nav',), ('fact_transactions',), ('fact_performance',)]


In [40]:
import pandas as pd

fund_df = pd.read_csv(
    "data/raw/01_fund_master.csv"
)

fund_df.to_sql(
    "dim_fund",
    conn,
    if_exists="append",
    index=False
)

print("Data Loaded")



Data Loaded


In [41]:
query = """
SELECT *
FROM dim_fund
LIMIT 5
"""

result = pd.read_sql(query, conn)

print(result)

   amfi_code       fund_house                                   scheme_name  \
0     119551  SBI Mutual Fund     SBI Bluechip Fund - Regular Plan - Growth   
1     119552  SBI Mutual Fund      SBI Bluechip Fund - Direct Plan - Growth   
2     119598  SBI Mutual Fund    SBI Small Cap Fund - Regular Plan - Growth   
3     119599  SBI Mutual Fund     SBI Small Cap Fund - Direct Plan - Growth   
4     119120  SBI Mutual Fund  SBI Magnum Gilt Fund - Regular Plan - Growth   

  category sub_category     plan launch_date                  benchmark  \
0   Equity    Large Cap  Regular  2006-02-14              NIFTY 100 TRI   
1   Equity    Large Cap   Direct  2013-01-01              NIFTY 100 TRI   
2   Equity    Small Cap  Regular  2009-09-09       BSE 250 SmallCap TRI   
3   Equity    Small Cap   Direct  2013-01-01       BSE 250 SmallCap TRI   
4     Debt         Gilt  Regular  2000-12-30  CRISIL Dynamic Gilt Index   

   expense_ratio_pct  exit_load_pct  min_sip_amount  min_lumpsum_amount  \

In [42]:
query = """
SELECT COUNT(*)
FROM dim_fund
"""

result = pd.read_sql(query, conn)

print(result)

   COUNT(*)
0        40


In [43]:
import sqlite3

conn = sqlite3.connect("bluestock_mf.db")

cursor = conn.cursor()

cursor.execute("SELECT sqlite_version();")

print(cursor.fetchone())

('3.42.0',)


In [44]:
import pandas as pd
from sqlalchemy import create_engine

# Database connection
engine = create_engine(
    "sqlite:///bluestock_mf.db"
)

# Read cleaned files
# fund_df = pd.read_csv(
#     "data/processed/dim_fund.csv"
# )

nav_df = pd.read_csv(
    "data/processed/02_nav_history_clean.csv"
)

txn_df = pd.read_csv(
    "data/processed/08_investor_transactions_clean.csv"
)

perf_df = pd.read_csv(
    "data/processed/07_scheme_performance_clean.csv"
)

# Load tables
fund_df.to_sql(
    "dim_fund",
    engine,
    if_exists="replace",
    index=False
)

nav_df.to_sql(
    "fact_nav",
    engine,
    if_exists="replace",
    index=False
)

txn_df.to_sql(
    "fact_transactions",
    engine,
    if_exists="replace",
    index=False
)

perf_df.to_sql(
    "fact_performance",
    engine,
    if_exists="replace",
    index=False
)

print("All data loaded successfully!")

All data loaded successfully!


In [45]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("bluestock_mf.db")

query = """
SELECT
strftime('%Y-%m', date) AS month,
AVG(nav) AS avg_nav
FROM fact_nav
GROUP BY month
"""

result = pd.read_sql(query, conn)

print(result)

      month     avg_nav
0   2022-01  207.061394
1   2022-02  207.717759
2   2022-03  209.692614
3   2022-04  211.833457
4   2022-05  212.731451
5   2022-06  213.860867
6   2022-07  213.956111
7   2022-08  215.683994
8   2022-09  218.494307
9   2022-10  219.529633
10  2022-11  223.470692
11  2022-12  226.760632
12  2023-01  230.671234
13  2023-02  233.847674
14  2023-03  238.009623
15  2023-04  240.613317
16  2023-05  241.892889
17  2023-06  244.605453
18  2023-07  245.780681
19  2023-08  247.734606
20  2023-09  251.833677
21  2023-10  255.607027
22  2023-11  258.352467
23  2023-12  262.020325
24  2024-01  264.620575
25  2024-02  266.331713
26  2024-03  269.597812
27  2024-04  271.300039
28  2024-05  273.450521
29  2024-06  275.465499
30  2024-07  279.024099
31  2024-08  279.736082
32  2024-09  281.016961
33  2024-10  282.127765
34  2024-11  285.243625
35  2024-12  285.272562
36  2025-01  288.038374
37  2025-02  291.744584
38  2025-03  296.158074
39  2025-04  299.412965
40  2025-05  303